In [1]:
import yaml
from jinja2 import Template
from langsmith import Client

### RAG Pipeline Prompt

In [ ]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [2]:
preprocessed_context = "- a \n- b"
question = "What is a?"

In [3]:
prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

In [4]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a?    



### Jinja Templates

In [5]:
jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}    
"""

In [6]:
template = Template(jinja_template)

In [7]:
rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)

In [8]:
print(rendered_template)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a?    


In [23]:
def build_prompt_jinja(preprocessed_context, question):    
    
    jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}    
    """

    template = Template(jinja_template)
    rendered_template = template.render(
        preprocessed_context=preprocessed_context,
        question=question
    )

    return rendered_template

In [10]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a?    
    


In [11]:
print(build_prompt_jinja("- Some item", "My silly question"))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- Some item

Question:
My silly question    
    


In [15]:
def prompt_template_config(yaml_path, prompt_key):

    with open(yaml_path, "r") as file:
        config = yaml.safe_load(file)

    template_content = config["prompts"][prompt_key]

    template = Template(template_content)

    return template

In [17]:
template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")

In [18]:
template

<Template memory:10c765ac0>

In [19]:
rendered_prompt = template.render(
    preprocessed_context=preprocessed_context,
    question=question
)

In [20]:
print(rendered_prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a? 


In [21]:
def build_prompt_jinja(preprocessed_context, question):

    template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")

    rendered_prompt = template.render(
        preprocessed_context=preprocessed_context,
        question=question
    )

    return rendered_prompt

In [22]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a? 


### Prompt Registries

In [24]:
ls_client = Client()

In [25]:
ls_template = ls_client.pull_prompt("retrieval-generation")

In [26]:
ls_template

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'retrieval-generation', 'lc_hub_commit_hash': '1b90e1125e075592c2432e66a7826cca4c575d36dd4ec316f5a8ba8a982f8660'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a shopping assistant that can answer questions about the products in stock.\nYou will be given a question and a list of context.\nInstructions:\n- Answer the question based on the provided context only.\n- Never use word context and refer to it as the available products.\n- Do not use markdown formatting.\nContext:\n{{ preprocessed_context }}\nQuestion:\n{{ question }}    '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [27]:
print(ls_template.messages[0].prompt.template)

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.
Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.
Context:
{{ preprocessed_context }}
Question:
{{ question }}    


In [28]:
def prompt_template_registry(prompt_name):

    template_content = ls_client.pull_prompt(prompt_name).messages[0].prompt.template

    template = Template(template_content)

    return template

In [29]:
print(
    prompt_template_registry("retrieval-generation").render(
        preprocessed_context=preprocessed_context,
        question=question
    )
)

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.
Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.
Context:
- a 
- b
Question:
What is a?    
